# E13 on CLERC: held-out passage queries, band (word) pipeline, CPU

The task: a passage of a case is the query, and the system must say which case it came from. The passage
is cut out of its case before indexing (`trr.data.heldout`), so it cannot be matched verbatim.

1. Builds the same CLERC pool as E11 / E12 (`scripts/build_clerc_pool.py`, deterministic).
2. Builds the held-out queries (`scripts/build_heldout_queries.py`): 2,000 cases, one query each, lengths
   [25, 100, 350] words; dev 20% / test 80% by case.
3. `scripts/run_band_pipeline.py --dataset clerc_heldout`: smoke -> dev search at 100 words -> one test run per length
   -> the intact sanity control (passages left in the index; full-query BM25 must find nearly all).
4. Packages tables, per-query runs, logs and the held-out query file.

As pre-registered (E13, `results/notes/experiment_log.md`). **Input:** private dataset `trr-code`.
**Settings:** Internet on; accelerator none (CPU). **Output:** `/kaggle/working/e13_band_outputs.zip`.

In [ ]:
import glob, os, shutil, subprocess, time
from pathlib import Path

def find(pattern, archive):
    # a path inside an input dataset; if Kaggle kept the uploaded zip packed, unpack it to /tmp first
    for root in ("/kaggle/input", "/tmp/inputs"):
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return Path(hits[0])
        zips = sorted(glob.glob(f"/kaggle/input/**/{archive}", recursive=True))
        if root == "/kaggle/input" and zips:
            shutil.unpack_archive(zips[0], f"/tmp/inputs/{archive[:-4]}")
    raise FileNotFoundError(f"{pattern} not found under /kaggle/input: is the trr-code dataset added?")

code_root = find("configs/default.yaml", "trr_code.zip").parents[1]

REPO = Path("/kaggle/working/trr")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(code_root, REPO)
subprocess.run(["chmod", "-R", "u+w", str(REPO)], check=True)        # inputs are read-only
(REPO / "data/cache").mkdir(parents=True, exist_ok=True)
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| git", (REPO / "GIT_HASH").read_text().strip())

DOWNLOAD = Path("/tmp/clerc_download")          # ~13 GB: outside /kaggle/working, so not saved as output
DOWNLOAD.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(DOWNLOAD).free / 1e9
print(f"free disk for the download: {free_gb:.0f} GB")
assert free_gb > 20, "need > 20 GB free under /tmp for the CLERC files"

In [ ]:
def run(cmd, log=None):
    """Run a shell command in the repo, stream its output, keep a log, stop the notebook on failure."""
    print("$", cmd, flush=True)
    t0, lines = time.time(), []
    with subprocess.Popen(cmd, shell=True, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            lines.append(line)
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[{time.time() - t0:.0f} s]", flush=True)
    if p.returncode:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

run("pip install -q -e '.[dev]'")
run("python -m pytest -q", log="e13_band_pytest.log")

## 1. CLERC pool and held-out queries

In [ ]:
run(f"python scripts/build_clerc_pool.py --download-dir {DOWNLOAD}", log="e13_band_pool.log")
run("python scripts/build_heldout_queries.py", log="e13_band_queries.log")

## 2. Smoke run (50 cases): catches errors in a minute

In [ ]:
run("python scripts/run_band_pipeline.py --dataset clerc_heldout --stage dev --smoke 50", log="e13_band_smoke_dev.log")
run("python scripts/run_band_pipeline.py --dataset clerc_heldout --stage test --smoke 50", log="e13_band_smoke_test.log")
run("python scripts/run_band_pipeline.py --dataset clerc_heldout --stage test --smoke 50 --intact", log="e13_band_smoke_intact.log")

## 3. Dev search at 100 words (dev queries only): picks the setting

In [ ]:
run("python scripts/run_band_pipeline.py --dataset clerc_heldout --stage dev", log="e13_band_dev.log")
print((REPO / "results/tables/band_dev_clerc_heldout_q100.md").read_text())

## 4. Test run, once per query length

In [ ]:
for qlen in [100, 25, 350]:
    run(f"python scripts/run_band_pipeline.py --dataset clerc_heldout --stage test --qlen {qlen}", log=f"e13_band_test_q{qlen}.log")
    print((REPO / f"results/tables/band_test_clerc_heldout_q{qlen}.md").read_text())

## 5. Sanity control: the same queries with the passages left in the index

In [ ]:
run("python scripts/run_band_pipeline.py --dataset clerc_heldout --stage test --qlen 100 --intact", log="e13_band_test_intact.log")
print((REPO / "results/tables/band_test_clerc_heldout_q100_intact.md").read_text())

## Package the outputs (tables, per-query runs, logs, held-out queries, pool metadata)

In [ ]:
import zipfile
out = Path("/kaggle/working/e13_band_outputs.zip")
patterns = ["results/tables/band_*clerc_heldout*", "results/runs/*_test_clerc_heldout*.jsonl",
            "results/logs/e13_band_*.log", "data/external/clerc/pool_*/heldout/*",
            "data/external/clerc/pool_*/pool.meta.json", "data/external/clerc/pool_*/queries.parquet"]
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in patterns:
        for f in sorted(REPO.glob(pattern)):
            z.write(f, f.relative_to(REPO))
            print(f.relative_to(REPO))
print(out, f"{out.stat().st_size / 1e6:.0f} MB")